# 卖家阶段整理

本课调用公共函数复现第 14 到第 16 份 Notebook 的卖家清洗、关联和汇总。Notebook 可从原始 CSV 独立运行。

| 导出文件 | 一行的含义 | 主键 |
| --- | --- | --- |
| sellers_clean_v1.csv | 一个卖家的地区属性及质量标记 | seller_id |
| items_with_sellers_v1.csv | 一条商品明细及对应卖家属性 | order_id + order_item_id |
| seller_profile_v1.csv | 一个有商品明细的卖家的销售汇总和地区属性 | seller_id |
| order_seller_pairs_v1.csv | 一个订单与一个卖家的关系及明细行数 | order_id + seller_id |

统计范围为全部商品明细及其覆盖的订单。商品金额与运费均以整数分核对。


## 1. 从原表读取

邮编前缀和 ID 按字符串读取，金额以原始文本交给明细清洗函数转换。


In [1]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

project_dir = Path.cwd()
if project_dir.name == 'notebooks':
    project_dir = project_dir.parent
if not (project_dir / 'src').is_dir():
    raise ValueError('请从项目根目录或 notebooks 目录打开本课')

data_dir = project_dir.parent / 'archive'
src_dir = str(project_dir / 'src')
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)

from items_cleaning import clean_order_items
from sellers_cleaning import (
    clean_sellers, join_items_sellers, summarize_seller_sales,
    build_order_seller_pairs, reconcile_seller_totals,
)

read_options = {
    'dtype': str, 'encoding': 'utf-8-sig',
    'keep_default_na': False, 'na_values': [''],
}
sellers_raw = pd.read_csv(data_dir / 'olist_sellers_dataset.csv', **read_options)
items_raw = pd.read_csv(data_dir / 'olist_order_items_dataset.csv', **read_options)


## 2. 清洗和关联

clean_sellers 返回一个卖家属性副本。clean_order_items 返回明细副本，并把金额转换为整数分。
join_items_sellers 验证多对一关系，返回附带卖家属性的明细。


In [2]:
sellers_clean = clean_sellers(sellers_raw)
items_clean = clean_order_items(items_raw)
items_with_sellers = join_items_sellers(items_clean, sellers_clean)

# 关联结果的粒度由左表决定，每行仍是一条商品明细。
display(items_with_sellers['sellers_match'].value_counts())
display(sellers_clean[['seller_id', 'seller_zip_code_prefix', 'seller_state']].head())


sellers_match
both          112650
left_only          0
right_only         0
Name: count, dtype: int64

,seller_id,seller_zip_code_prefix,seller_state
0,3442f8959a84dea7ee197c632cb2df15,13023,SP
1,d1b65fc7debc3361ea86b5f14c68d2e2,13844,SP
2,ce3ad9de960102d0677a81f5d0bb7b2d,20031,RJ
3,c0f3eea2e14555b6faeea3dd58c1b1c3,04195,SP
4,51a04a8a6bdcb23deccc82b0b80742cf,12914,SP


## 3. 生成两种汇总

seller_profile 按卖家统计明细行数、不同订单数、不同商品数及金额。
order_seller_pairs 按订单和卖家的组合统计该组合有多少条明细。


In [3]:
seller_profile = summarize_seller_sales(items_with_sellers, sellers_clean)
order_seller_pairs = build_order_seller_pairs(items_with_sellers)

display(seller_profile[[
    'seller_id', 'seller_state', 'item_line_count',
    'order_count', 'product_count', 'goods_amount_cents', 'freight_amount_cents',
]].head())
display(order_seller_pairs.head())


,seller_id,seller_state,item_line_count,order_count,product_count,goods_amount_cents,freight_amount_cents
0,0015a82c2db000af6aaaf3ae2ecb0532,SP,3,3,1,268500,6306
1,001cca7ae9ae17fb1caed9dfb1094831,ES,239,200,11,2508003,885414
2,001e6ad469a905060d959994f1b41e4f,RJ,1,1,1,25000,1794
3,002100f778ceb8431b7a1020ff7ab48f,SP,55,51,24,123450,79366
4,003554e2dce176b5555353e4f3555ac8,GO,1,1,1,12000,1938


,order_id,seller_id,item_line_count
0,00010242fe8c5a6d1ba2dd792cb16214,48436dade18ac8b2bce089ec2a041202,1
1,00018f77f2f0320c557190d7a144bdd3,dd7ddc04e1b6c2c614352b383efe2d36,1
2,000229ec398224ef6ca0657da4fc703e,5b51032eddd242adc84c38acab88f23d,1
3,00024acbcdf0a6daa1e931b038114c75,9d7a1d34a5052409006425275ba1c2b4,1
4,00042b26cf59d7ce69dfabb4e55b4fd9,df560393f3a51e74553ab94004ba5c87,1


## 4. 核对两种订单计数

明细数与金额从销售汇总加总核对。各卖家订单数之和用关系表行数核对。
全局订单数用不同的订单 ID 数核对。


In [4]:
seller_controls = reconcile_seller_totals(items_clean, seller_profile, order_seller_pairs)
display(seller_controls)

# 关系表已有组合主键，所以每个订单组的行数就是该订单涉及的卖家数。
sellers_per_order = order_seller_pairs.groupby('order_id').size()
display(pd.Series({
    '全体不同订单数': sellers_per_order.size,
    '订单＋卖家关系数': len(order_seller_pairs),
    '多卖家订单数': sellers_per_order.gt(1).sum(),
    '额外关系数': (sellers_per_order - 1).sum(),
}))


,应有值,实际值,差额
卖家汇总明细数,112650,112650,0
卖家汇总商品金额（分）,1359164370,1359164370,0
卖家汇总运费（分）,225190954,225190954,0
各卖家订单数之和,100010,100010,0
关系表明细数,112650,112650,0
关系表不同订单数,98666,98666,0


全体不同订单数      98666
订单＋卖家关系数    100010
多卖家订单数        1278
额外关系数         1344
dtype: int64

## 5. 从终端导出

在项目根目录的 PowerShell 终端运行以下命令。

    python .\src\08_clean_sellers.py

四张 CSV 位于 data/processed/sellers_v1。控制表、每单卖家数分布和阶段报告位于 reports/sellers_v1。
再次读入 CSV 时，将 ID 与邮编前缀指定为字符串。

本课的核对任务是运行脚本，确认卖家汇总有 3,095 行，关系表有 100,010 行，六项控制差额均为 0。
下一课检查地理表的粒度、坐标与邮编覆盖。
